# F26-13 Phase 3 on Kaggle

Same pipeline as `04_phase3_colab.ipynb`, made for Kaggle so long runs keep going after the browser is closed.

Settings: **Accelerator → GPU T4 x2**, **Internet → On**. If you have `best.pt` and the detection caches, upload them as a dataset and add it as input; the notebook finds them on its own. Otherwise it trains YOLO and builds the caches itself.

Then **Save Version → Save & Run All (Commit)**. Results are in the **Output** tab: `runs_phase3.zip`, `masks_kvasir_test.zip` and `tables/`.

In [ ]:
!nvidia-smi

## Code and packages

In [ ]:
import os, glob, json, shutil, subprocess, time
PROJ = "/tmp/proj"
OUT = "/kaggle/working"

!rm -rf {PROJ}
!git clone -q https://github.com/kashishfatima999/Confidence-Aware-Self-Prompting-for-Robust-Polyp-Segmentation.git {PROJ}
%cd {PROJ}
!git log -1 --format="commit %h %s"

In [ ]:
!pip install -q -r requirements-colab.txt kagglehub
!SAM2_BUILD_CUDA=0 pip install -q "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4"
!pip freeze | grep -i -E "^torch==|^ultralytics==|SAM-2"
!python -m unittest discover -s tests 2>&1 | tail -3

## What to run

In [ ]:
SMOKE = False
RUN_GRID = False
RUN_TESTS = True
RUN_CORRUPTED = True

LIMIT = "--limit 5" if SMOKE else ""

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-4000:])
        raise RuntimeError(cmd)
    return r.stdout

## Data

In [ ]:
import kagglehub
import pandas as pd
from PIL import Image

p = kagglehub.dataset_download("debeshjha1/kvasirseg")
src = f"{p}/Kvasir-SEG/Kvasir-SEG"
dest = "data/raw/kvasir-seg/Kvasir-SEG"
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(f"{src}/images", f"{dest}/images")
shutil.copytree(f"{src}/masks", f"{dest}/masks")

boxes = {}
for csv_path in sorted(glob.glob(f"{src}/bbox/*.csv")):
    image_id = os.path.splitext(os.path.basename(csv_path))[0]
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    w, h = Image.open(f"{dest}/images/{image_id}.jpg").size
    boxes[image_id] = {"height": h, "width": w,
                       "bbox": [{"label": "polyp", "xmin": int(r["xmin"]), "ymin": int(r["ymin"]),
                                 "xmax": int(r["xmax"]), "ymax": int(r["ymax"])} for _, r in df.iterrows()]}
with open(f"{dest}/kavsir_bboxes.json", "w") as f:
    json.dump(boxes, f)
print("boxes:", len(boxes))

In [ ]:
!python src/data/prepare_kvasir.py | head -3
p2 = kagglehub.dataset_download("balraj98/cvcclinicdb")
shutil.rmtree("data/raw/cvc-clinicdb", ignore_errors=True)
shutil.copytree(f"{p2}/PNG", "data/raw/cvc-clinicdb/PNG")
!python src/data/prepare_clinicdb.py | head -2
!python src/data/make_corruptions.py | tail -1

## Weights

In [ ]:
os.makedirs("weights", exist_ok=True)
!wget -q -nc -P weights https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

found = glob.glob("/kaggle/input/**/best.pt", recursive=True)
if found:
    BEST = found[0]
    print("using", BEST)
else:
    print("no best.pt in the inputs, training YOLO")
    sh("python src/train_yolo.py" + (" --epochs 1 --fraction 0.1" if SMOKE else ""))
    BEST = "runs/yolo/yolov8m_kvasir_s42/weights/best.pt"
    shutil.copy(BEST, OUT)
    shutil.copy("runs/yolo/yolov8m_kvasir_s42/summary.json", f"{OUT}/yolo_summary.json")

## Detection cache

In [ ]:
CORRUPTED = [f"kvasir_corrupted/{c}_s{s}" for c in ["blur", "brightness_contrast", "noise"] for s in (1, 2, 3)]
os.makedirs("runs/detections", exist_ok=True)

def get_cache(ds):
    name = ds.replace("/", "_") + ".json"
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if found:
        shutil.copy(found[0], "runs/detections/")
        how = "from input"
    else:
        sh(f"python src/pipeline/detect.py --weights {BEST} --dataset {ds} {LIMIT}")
        how = "built"
    n = json.load(open(f"runs/detections/{name}"))["images"]
    print(f"{ds}: {len(n)} images ({how})")

for ds in ["kvasir_val", "kvasir_test", "clinicdb"]:
    get_cache(ds)

## Runs

In [ ]:
def run(name, ds, config, extra=""):
    out = f"runs/{name}/{ds.replace('/', '_')}"
    t = time.time()
    if not os.path.exists(f"{out}/summary.json"):
        sh(f"python src/pipeline/run_experiment.py --config configs/experiments/{config}.yaml "
           f"--dataset {ds} --name {name} {extra} {LIMIT}")
    s = json.load(open(f"{out}/summary.json"))
    print(f"{name:28s} {ds:36s} dice {s['mean_dice']:.4f}  missed {s['images_with_empty_prediction']:3d}  "
          f"fp {s['detection_fp']:3d}  {time.time() - t:.0f}s", flush=True)
    return s

In [ ]:
if RUN_GRID:
    rows = []
    for t_low in [0.15, 0.25, 0.35]:
        for t_high in [0.6, 0.7, 0.8]:
            for s_min in [0.6, 0.7, 0.8]:
                s = run(f"grid/tl{t_low}_th{t_high}_s{s_min}", "kvasir_val", "casp",
                        f"--set t_low={t_low} t_high={t_high} s_min={s_min}")
                rows.append({"t_low": t_low, "t_high": t_high, "s_min": s_min, "mean_dice": s["mean_dice"],
                             "mean_iou": s["mean_iou"], "missed": s["images_with_empty_prediction"],
                             "fp": s["detection_fp"], "prompts_per_image": s["prompts_per_image"]})
    run("grid/baseline", "kvasir_val", "baseline")
    grid = pd.DataFrame(rows).sort_values(["mean_dice", "fp"], ascending=[False, True])
    grid.to_csv(f"{OUT}/casp_val_grid.csv", index=False)
    print(grid.head(10).to_string(index=False))

In [ ]:
EXPERIMENTS = {
    "baseline": "--save-masks",
    "threshold_only": "",
    "casp": "--save-masks",
    "casp_all": "",
    "casp_noiter": "",
    "oracle": "",
}

if RUN_TESTS:
    for ds in ["kvasir_test", "clinicdb"]:
        for name, extra in EXPERIMENTS.items():
            run(name, ds, name, extra)

In [ ]:
if RUN_CORRUPTED:
    for ds in CORRUPTED:
        get_cache(ds)
        run("baseline", ds, "baseline")
        run("casp", ds, "casp")

## Tables and output

In [ ]:
sh("python src/eval/summarize.py")
shutil.copytree("docs/tables/phase3", f"{OUT}/tables", dirs_exist_ok=True)
sh(f"cd runs && zip -qr {OUT}/runs_phase3.zip . -x '*/masks/*' 'yolo/*'")
if os.path.exists("runs/casp/kvasir_test/masks"):
    sh(f"cd runs && zip -qr {OUT}/masks_kvasir_test.zip casp/kvasir_test/masks baseline/kvasir_test/masks")
print(open("docs/tables/phase3/results.md").read())
!ls -la {OUT}